# 1. Préparation des Données et Extraction TF-IDF
Chargement des fichiers nettoyés, découpage 80/20 et vectorisation du texte des e-mails.


In [ ]:
import os
import sys
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Importation de nos modules personnalisés
sys.path.append('..')
from src.features import extraire_indices_url, nettoyer_texte_email
from src.models import preparer_et_decouper_donnees, entrainer_modele_xgb, entrainer_modele_rf

DOSSIER_BRUT = "../data/raw"

# --- 1. TRAITEMENT DES URLS ---
print("Traitement du dataset URLs...")
chemin_urls = os.path.join(DOSSIER_BRUT, "PhiUSIIL_Phishing_URL_Dataset.csv")
df_urls = pd.read_csv(chemin_urls)

X_url_train_brut, X_url_test_brut, y_url_train, y_url_test = preparer_et_decouper_donnees(
    df_urls, colonne_texte='URL', colonne_cible='label'
)

# Application de notre extracteur de features cyber
print("Extraction des indices cyber sur les URLs...")
X_url_train = pd.DataFrame([extraire_indices_url(x) for x in X_url_train_brut])
X_url_test = pd.DataFrame([extraire_indices_url(x) for x in X_url_test_brut])


# --- 2. TRAITEMENT DES EMAILS ---
print("\nTraitement du dataset Emails...")
chemin_emails = os.path.join(DOSSIER_BRUT, "phishings_emails/phishing_email.csv")
df_emails = pd.read_csv(chemin_emails)

X_email_train_brut, X_email_test_brut, y_email_train, y_email_test = preparer_et_decouper_donnees(
    df_emails, colonne_texte='text_combined', colonne_cible='label'
)

# Nettoyage NLP et transformation vectorielle TF-IDF
print("Nettoyage NLP et vectorisation TF-IDF des emails...")
X_email_train_nettoye = X_email_train_brut.apply(nettoyer_texte_email)
X_email_test_nettoye = X_email_test_brut.apply(nettoyer_texte_email)

# Limitation à 5000 mots fréquents pour optimiser la mémoire et la vitesse
vectoriseur = TfidfVectorizer(max_features=5000, stop_words='english')
X_email_train = vectoriseur.fit_transform(X_email_train_nettoye)
X_email_test = vectoriseur.transform(X_email_test_nettoye)

print("\nDimensions des matrices d'entrainement :")
print(f"URLs  Train : {X_url_train.shape}")
print(f"Emails Train : {X_email_train.shape}")


ModuleNotFoundError: No module named 'xgboost'

# 2. Entraînement des Modèles de Machine Learning
Lancement de l'apprentissage avec Random Forest et XGBoost sur les deux problématiques.


In [ ]:
print("Entrainement des modeles pour les URLs...")
modele_url_rf = entrainer_modele_rf(X_url_train, y_url_train)
modele_url_xgb = entrainer_modele_xgb(X_url_train, y_url_train)

print("\nEntrainement des modeles pour les Emails (cette etape peut prendre 1 a 2 minutes)...")
modele_email_rf = entrainer_modele_rf(X_email_train, y_email_train)
modele_email_xgb = entrainer_modele_xgb(X_email_train, y_email_train)

print("\nTous les modeles ont ete entraines avec succes.")


# 3. Évaluation Orientée Cybersécurité (Matrice de Confusion et Recall)
Analyse des performances des modèles avec un focus critique sur le Recall pour minimiser les Faux Négatifs.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluer_performances_cyber(modele, X_test, y_test, nom_modele):
    """
    Affiche la matrice de confusion et le rapport de classification epure.
    """
    predictions = modele.predict(X_test)
    matrice = confusion_matrix(y_test, predictions)
    rapport = classification_report(y_test, predictions, output_dict=True)
    
    # Extraction des indicateurs critiques
    faux_negatifs = matrice[1][0]
    faux_positifs = matrice[0][1]
    rappel_phishing = rapport['1']['recall'] * 100
    
    print(f"--- EVALUATION : {nom_modele} ---")
    print(f"Recall (Taux de detection du Phishing) : {rappel_phishing:.2f}%")
    print(f"Faux Negatifs (Phishing non detectes - DANGER) : {faux_negatifs}")
    print(f"Faux Positifs (Sains bloques par erreur) : {faux_positifs}\n")

# Evaluation de la partie URLs
evaluer_performances_cyber(modele_url_rf, X_url_test, y_url_test, "Random Forest (URLs)")
evaluer_performances_cyber(modele_url_xgb, X_url_test, y_url_test, "XGBoost (URLs)")

# Evaluation de la partie Emails
evaluer_performances_cyber(modele_email_rf, X_email_test, y_email_test, "Random Forest (Emails)")
evaluer_performances_cyber(modele_email_xgb, X_email_test, y_email_test, "XGBoost (Emails)")
